# 🎹 Aria — Chopin LoRA (Merged) Generator

**Workflow:**
1. Run Step 1 (install)
2. Run Step 2 (upload LoRA zip)
3. Run Step 3 (merge at chosen strength)
4. Run Step 4 (load merged model)
5. Run Step 5 (generate) — rerun freely
6. Run Step 6 (listen)

**To try a different strength:** Run the Clear cell, then rerun Step 3 with new `LORA_STRENGTH`, then Step 4, then Step 5.

**Runtime:** T4 GPU

## Step 1 — Install
**Run once per session.**

In [ ]:
!pip install -q "torchao>=0.16.0"
!pip uninstall -q -y transformers
!pip install -q "git+https://github.com/EleutherAI/aria.git#egg=aria[all]"
!pip install -q huggingface_hub peft safetensors accelerate midi2audio
!apt-get install -qq fluidsynth
!wget -q -O /tmp/piano2.sf2 https://github.com/urish/cinto/raw/master/media/FluidR3_GM.sf2

import os, torch, gc, glob
from datetime import datetime
from aria.run import _get_prompt
from aria.inference.sample_cuda import sample_batch
from ariautils.tokenizer import AbsTokenizer

os.makedirs('aria_weights', exist_ok=True)
os.makedirs('generated_midi', exist_ok=True)

print(f'✅ Ready — GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Upload LoRA ZIP
**Upload your LoRA zip file once. No need to re-upload when changing strength.**

In [ ]:
from google.colab import files as colab_files
import zipfile

print('Upload your LoRA zip file...')
uploaded = colab_files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('lora_extracted')

LORA_PATH = None
for root, dirs, filenames in os.walk('lora_extracted'):
    if 'adapter_config.json' in filenames:
        LORA_PATH = root
        break

if LORA_PATH is None:
    print('❌ adapter_config.json not found — check your zip')
else:
    print(f'✅ LoRA ready at: {LORA_PATH}')
    print(f'Files: {os.listdir(LORA_PATH)}')

## Step 3 — Merge LoRA at Chosen Strength
**Change `LORA_STRENGTH` and rerun to try different values.**
Each strength is saved as a separate file.

In [ ]:
from transformers import AutoModelForCausalLM, AutoConfig
from peft import PeftModel
import transformers.tokenization_utils as _tok_utils
from transformers.tokenization_utils_base import BatchEncoding
from safetensors.torch import load_file, save_file
from huggingface_hub import hf_hub_download

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

LORA_STRENGTH = 1.0   # 0.5 = subtle | 1.0 = full | 1.5 = exaggerated

# ══════════════════════════════════════════

if not hasattr(_tok_utils, 'BatchEncoding'):
    _tok_utils.BatchEncoding = BatchEncoding

print('Loading base model + gen checkpoint...')
config = AutoConfig.from_pretrained('loubb/aria-medium-base', trust_remote_code=True)
if not hasattr(config, 'initializer_range'):
    config.initializer_range = 0.02

hf_base = AutoModelForCausalLM.from_pretrained(
    'loubb/aria-medium-base', config=config, trust_remote_code=True, dtype=torch.float16
)
gen_ckpt = hf_hub_download(repo_id='loubb/aria-medium-base', filename='model-gen.safetensors')
hf_base.load_state_dict(load_file(gen_ckpt), strict=False)

print(f'Applying LoRA at strength {LORA_STRENGTH}...')
lora_model = PeftModel.from_pretrained(hf_base, LORA_PATH)
for module in lora_model.modules():
    if hasattr(module, 'scaling') and isinstance(module.scaling, dict):
        for key in module.scaling:
            module.scaling[key] *= LORA_STRENGTH

print('Merging...')
merged = lora_model.merge_and_unload()

MERGED_PATH = f'aria_weights/chopin_s{LORA_STRENGTH}.safetensors'
save_file(merged.state_dict(), MERGED_PATH)

del merged, lora_model, hf_base
gc.collect()
torch.cuda.empty_cache()

print(f'✅ Saved: {MERGED_PATH}')
print(f'Now run Step 4 to load it.')

## Step 4 — Load Merged Model
**Run after Step 3. Uses the merged file from the strength you just set.**

In [ ]:
from aria.run import _load_inference_model_torch

model = _load_inference_model_torch(
    checkpoint_path=MERGED_PATH,
    config_name='medium',
    strict=False,
)
print(f'✅ Loaded: {MERGED_PATH}')
print(f'KV cache enabled — fast generation ready')

## Step 5 — Upload Seed & Generate
**Rerun this cell freely with different settings.**

In [ ]:
from google.colab import files as colab_files

print('Upload your seed MIDI file:')
uploaded = colab_files.upload()
SEED_MIDI_PATH = list(uploaded.keys())[0]
print(f'✅ Seed: {SEED_MIDI_PATH}')

In [ ]:
tokenizer = AbsTokenizer()

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

SEED_DURATION_SEC = 8
MAX_NEW_TOKENS    = 4096    # ~2-3 min of music
TEMPERATURE       = 1.1
MIN_P             = 0.032
NUM_VARIATIONS    = 2       # generated in parallel — same time as 1!

# ══════════════════════════════════════════

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
prompt = _get_prompt(SEED_MIDI_PATH, prompt_duration_s=SEED_DURATION_SEC)
print(f'Generating {NUM_VARIATIONS} variations in parallel...\n')

results = sample_batch(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    num_variations=NUM_VARIATIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    temp=TEMPERATURE,
    min_p=MIN_P,
)

for idx, seq in enumerate(results):
    midi_dict = tokenizer.detokenize(seq)
    out_path = f'generated_midi/chopin_s{LORA_STRENGTH}_{timestamp}_{idx+1}.mid'
    midi_dict.to_midi().save(out_path)
    print(f'✅ Saved: {out_path}')

## Step 6 — Listen

In [ ]:
from IPython.display import Audio, display

for midi_path in sorted(glob.glob('generated_midi/*.mid')):
    wav_path = midi_path.replace('.mid', '.wav')
    mp3_path = midi_path.replace('.mid', '.mp3')
    os.system(f'fluidsynth -ni /tmp/piano2.sf2 "{midi_path}" -F "{wav_path}" -r 16000')
    os.system(f'ffmpeg -loglevel quiet -i "{wav_path}" "{mp3_path}"')
    print('🎹', os.path.basename(midi_path))
    display(Audio(mp3_path))

## Step 7 — Download

In [ ]:
from google.colab import files as colab_files
import zipfile

zip_path = '/content/generated_midis.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in glob.glob('generated_midi/*.mid'):
        zipf.write(f)
print('✅ ZIP created')
colab_files.download(zip_path)

---
## Utilities

In [ ]:
# ── Clear model from GPU — run before trying a new strength ──────────
# After running this: rerun Step 3 with new LORA_STRENGTH, then Step 4, then Step 5

try:
    del model
    print('Model cleared')
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print('✅ GPU cleared — rerun Step 3 with new LORA_STRENGTH')

In [ ]:
# ── Delete generated MIDIs ───────────────────────────────────────────
for f in glob.glob('generated_midi/*'):
    os.remove(f)
print('🗑️ Generated MIDIs cleared')